# Навчання захисту від підробки в Colab

Увімкніть GPU: *Runtime → Change runtime type → T4 GPU*. Дані беруться з дзеркала CelebA-Spoof і, за бажанням, з вашої папки `custom` на Google Диску.

In [ ]:
!git clone -b improvements https://github.com/Olekselot/Zapus-na-zochut.git
%cd Zapus-na-zochut
!pip -q install -r requirements-data.txt onnxruntime-gpu

In [ ]:
!python tools/prepare_celeba_spoof.py --split train --shards 10 --max-live 16000 --max-spoof 16000
!python tools/prepare_celeba_spoof.py --split test --shards 4 --max-live 2000 --max-spoof 2000

Свій набір (необов'язково): покладіть на Диск папку `custom` зі структурою `live/p01/*.jpg`, `print/p01/*.jpg`, `phone/...`, `laptop/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data && cp -r /content/drive/MyDrive/custom data/custom 2>/dev/null; ls data

In [ ]:
!python training/train_antispoof.py --data data/celeba_spoof --epochs 15 --balance --batch-size 128 --workers 2 --run-name colab

In [ ]:
!python eval/antispoof_metrics.py --data data/celeba_spoof_test --split all
!test -d data/custom && python eval/antispoof_metrics.py --data data/custom --split all --name antispoof_custom_cross

In [ ]:
!mkdir -p /content/drive/MyDrive/zapus_models && cp models/antispoof.pt models/antispoof_meta.json eval/results/*.json eval/results/*.png /content/drive/MyDrive/zapus_models/